In [1]:
!apt-get install swig
!pip install gymnasium[box2d]
!pip install gym
!pip install pygame

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  swig4.0
Suggested packages:
  swig-doc swig-examples swig4.0-examples swig4.0-doc
The following NEW packages will be installed:
  swig swig4.0
0 upgraded, 2 newly installed, 0 to remove and 49 not upgraded.
Need to get 1,116 kB of archives.
After this operation, 5,542 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 swig4.0 amd64 4.0.2-1ubuntu1 [1,110 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 swig all 4.0.2-1ubuntu1 [5,632 B]
Fetched 1,116 kB in 2s (678 kB/s)
Selecting previously unselected package swig4.0.
(Reading database ... 123632 files and directories currently installed.)
Preparing to unpack .../swig4.0_4.0.2-1ubuntu1_amd64.deb ...
Unpacking swig4.0 (4.0.2-1ubuntu1) ...
Selecting previously unselected package swig.
Preparing to unpack .../swig_4.0.2-1ubunt

In [ ]:
import os
import warnings
import logging
from contextlib import redirect_stdout

# Suppress TensorFlow logs
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings("ignore")

# Suppress gym logs
logging.getLogger('gym').setLevel(logging.CRITICAL)

import numpy as np
import random
from collections import deque
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, BatchNormalization, Dropout
from tensorflow.keras.optimizers import Adam

class DQNAgent:
    def __init__(self, state_size, action_size):
        self.state_size = state_size
        self.action_size = action_size

        # Hyperparameters
        self.LEARNING_RATE = 0.001
        self.DISCOUNT = 0.95
        self.EPSILON = 1.0
        self.EPSILON_DECAY = 0.995
        self.EPSILON_MIN = 0.01
        self.MEMORY_SIZE = 100000
        self.MIN_REPLAY_MEMORY_SIZE = 1000
        self.MINIBATCH_SIZE = 64
        self.replay_memory = deque(maxlen=self.MEMORY_SIZE)
        self.model = self.build_dqn()

    def build_dqn(self):
        model = Sequential()
        model.add(Dense(128, input_dim=self.state_size, activation='relu'))
        model.add(Dense(128, activation='relu'))
        model.add(BatchNormalization())
        model.add(Dense(128, activation='relu'))
        model.add(Dropout(0.2))
        model.add(Dense(self.action_size, activation='linear'))
        model.compile(loss='mse', optimizer=Adam(learning_rate=self.LEARNING_RATE))
        return model

    def update_replay_memory(self, transition):
        self.replay_memory.append(transition)

    def act(self, state):
        q_values = self.model.predict(state[np.newaxis], verbose=0)  # Ensure no output
        if np.random.rand() < self.EPSILON:
            return random.choice(range(self.action_size))
        return np.argmax(q_values[0])

    def train(self):
        if len(self.replay_memory) < self.MIN_REPLAY_MEMORY_SIZE:
            return

        minibatch = random.sample(self.replay_memory, self.MINIBATCH_SIZE)

        current_states = np.array([transition[0] for transition in minibatch])
        actions = np.array([transition[1] for transition in minibatch])
        rewards = np.array([transition[2] for transition in minibatch])
        next_states = np.array([transition[3] for transition in minibatch])
        dones = np.array([transition[4] for transition in minibatch])

        current_qs = self.model.predict(current_states, verbose=0)
        future_qs = self.model.predict(next_states, verbose=0)

        for i in range(self.MINIBATCH_SIZE):
            if not dones[i]:
                current_qs[i][actions[i]] = rewards[i] + self.DISCOUNT * np.max(future_qs[i])
            else:
                current_qs[i][actions[i]] = rewards[i]

        self.model.fit(current_states, current_qs, batch_size=self.MINIBATCH_SIZE, verbose=0)

        if self.EPSILON > self.EPSILON_MIN:
            self.EPSILON *= self.EPSILON_DECAY

import gym

env = gym.make('LunarLander-v2')
state_size = env.observation_space.shape[0]
action_size = env.action_space.n

agent = DQNAgent(state_size, action_size)

EPISODES = 50000
SHOW_EVERY = 3000

with open(os.devnull, 'w') as fnull:
    with redirect_stdout(fnull):  # Suppress all unwanted outputs
        for episode in range(1, EPISODES + 1):
            print('episode: ',episode)
            current_state = env.reset()
            done = False
            total_reward = 0

            while not done:
                action = agent.act(current_state)
                new_state, reward, done, info = env.step(action)
                agent.update_replay_memory((current_state, action, reward, new_state, done))
                agent.train()
                current_state = new_state
                total_reward += reward

            if episode % SHOW_EVERY == 0:
                print(f"Episode: {episode}, Reward: {total_reward}, Epsilon: {agent.EPSILON:.4f}")


/usr/local/lib/python3.10/dist-packages/pygame/pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists
/usr/local/lib/python3.10/dist-packages/pkg_resources/__init__.py:3154: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('google')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-packages
  declare_namespace(pkg)
/usr/local/lib/python3.10/dist-packages/pkg_resources/__init__.py:3154: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('google.cloud')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-pa

In [13]:
!pip install torch gymnasium numpy stable-baselines3 "shimmy>=2.0"

import os
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import random
from collections import deque
from stable_baselines3.common.vec_env import DummyVecEnv


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

#  Create Vectorized Environment
NUM_ENVS = 4  # Run 4 environments in parallel
env = DummyVecEnv([lambda: gym.make("LunarLander-v3") for _ in range(NUM_ENVS)])

state_size = env.observation_space.shape[0]
action_size = env.action_space.n


LEARNING_RATE = 0.001
DISCOUNT = 0.99
EPSILON = 1.0
EPSILON_DECAY = 0.995
EPSILON_MIN = 0.05
MEMORY_SIZE = 100000
BATCH_SIZE = 64
TARGET_UPDATE_FREQ = 1000
TOTAL_TIMESTEPS = 500000
TRAINING_START = 10000

# Experience Replay Memory
class ReplayBuffer:
    def __init__(self, size):
        self.buffer = deque(maxlen=size)

    def add(self, experience):
        self.buffer.append(experience)

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            torch.tensor(np.array(states), dtype=torch.float32, device=device),
            torch.tensor(actions, dtype=torch.int64, device=device),
            torch.tensor(rewards, dtype=torch.float32, device=device),
            torch.tensor(np.array(next_states), dtype=torch.float32, device=device),
            torch.tensor(dones, dtype=torch.float32, device=device),
        )

    def can_sample(self, batch_size):
        return len(self.buffer) >= batch_size

memory = ReplayBuffer(MEMORY_SIZE)

# Build Q-Network
class DQN(nn.Module):
    def __init__(self, state_size, action_size):
        super(DQN, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(state_size, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, action_size)
        )

    def forward(self, x):
        return self.network(x)

# Create two networks: online model and target model
model = DQN(state_size, action_size).to(device)
target_model = DQN(state_size, action_size).to(device)
target_model.load_state_dict(model.state_dict())  # Copy weights
target_model.eval()  # Target network is not trained

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
loss_fn = nn.MSELoss()

# Training function (Batch Training like SB3)
def train():
    if not memory.can_sample(BATCH_SIZE):
        return

    states, actions, rewards, next_states, dones = memory.sample(BATCH_SIZE)

    # Compute target Q-values
    with torch.no_grad():
        target_qs = target_model(next_states)
        max_future_q = torch.max(target_qs, dim=1)[0]
        new_qs = rewards + (1 - dones) * DISCOUNT * max_future_q

    # Get current Q-values and update only selected actions
    current_qs = model(states)
    current_qs = current_qs.gather(1, actions.unsqueeze(1)).squeeze(1)

    # Compute loss and update model
    loss = loss_fn(current_qs, new_qs)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item()  # Return loss value for tracking


# Training Loop (SB3-like strategy) with Metrics
state = env.reset()
state = torch.tensor(state, dtype=torch.float32, device=device)

step = 0
episode_rewards = []
episode_lengths = []
current_episode_reward = np.zeros(NUM_ENVS)
current_episode_length = np.zeros(NUM_ENVS)

while step < TOTAL_TIMESTEPS:
    if np.random.rand() < EPSILON:
        action = np.array([env.action_space.sample() for _ in range(NUM_ENVS)])  # Random actions
    else:
        with torch.no_grad():
            q_values = model(state)
            action = torch.argmax(q_values, dim=1).cpu().numpy()  # Select best actions

    # Take action and observe result
    next_state, reward, done, _ = env.step(action)
    next_state = torch.tensor(next_state, dtype=torch.float32, device=device)

    for i in range(NUM_ENVS):
        memory.add((state[i].cpu().numpy(), action[i], reward[i], next_state[i].cpu().numpy(), done[i]))

        current_episode_reward[i] += reward[i]
        current_episode_length[i] += 1

        if done[i]:  # End of episode
            episode_rewards.append(current_episode_reward[i])
            episode_lengths.append(current_episode_length[i])
            current_episode_reward[i] = 0
            current_episode_length[i] = 0

    state = next_state
    step += NUM_ENVS  # Increment step by number of envs

    # Train only after collecting enough data
    loss = None
    if step > TRAINING_START and step % 4 == 0:
        loss = train()

    # Update target network periodically
    if step % TARGET_UPDATE_FREQ == 0:
        target_model.load_state_dict(model.state_dict())

    # Decay exploration rate
    if EPSILON > EPSILON_MIN:
        EPSILON *= EPSILON_DECAY

    # Print progress every 10,000 steps
    if step % 10000 == 0:
        avg_reward = np.mean(episode_rewards[-100:]) if len(episode_rewards) > 0 else 0
        avg_length = np.mean(episode_lengths[-100:]) if len(episode_lengths) > 0 else 0
        last_loss = loss if loss is not None else 0

        print(f"Step: {step}, Epsilon: {EPSILON:.4f}, Avg Reward: {avg_reward:.2f}, Avg Length: {avg_length:.1f}, Loss: {last_loss:.4f}")

# Save trained model
torch.save(model.state_dict(), "dqn_lunarlander_pytorch.pth")

print("-->>Training complete! Model saved as 'dqn_lunarlander_pytorch.pth'.")



Using device: cuda
Step: 10000, Epsilon: 0.0499, Avg Reward: -173.82, Avg Length: 72.9, Loss: 0.0000
Step: 20000, Epsilon: 0.0499, Avg Reward: -210.38, Avg Length: 121.8, Loss: 222.1492
Step: 30000, Epsilon: 0.0499, Avg Reward: -209.47, Avg Length: 212.8, Loss: 2.2447
Step: 40000, Epsilon: 0.0499, Avg Reward: -195.46, Avg Length: 292.2, Loss: 3.3039
Step: 50000, Epsilon: 0.0499, Avg Reward: -178.31, Avg Length: 385.0, Loss: 3.1276
Step: 60000, Epsilon: 0.0499, Avg Reward: -132.91, Avg Length: 464.5, Loss: 4.1650
Step: 70000, Epsilon: 0.0499, Avg Reward: -125.56, Avg Length: 567.1, Loss: 1.8163
Step: 80000, Epsilon: 0.0499, Avg Reward: -100.13, Avg Length: 640.1, Loss: 2.7985
Step: 90000, Epsilon: 0.0499, Avg Reward: -89.02, Avg Length: 706.0, Loss: 37.0001
Step: 100000, Epsilon: 0.0499, Avg Reward: -81.84, Avg Length: 753.6, Loss: 5.2264
Step: 110000, Epsilon: 0.0499, Avg Reward: -76.72, Avg Length: 756.3, Loss: 1.8067
Step: 120000, Epsilon: 0.0499, Avg Reward: -77.87, Avg Length: 760.

In [15]:
!pip install torch gymnasium numpy stable-baselines3 "shimmy>=2.0"

import os
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import random
from collections import deque
from stable_baselines3.common.vec_env import DummyVecEnv


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

#  Create Vectorized Environment
NUM_ENVS = 16 # Run 4 environments in parallel
env = DummyVecEnv([lambda: gym.make("LunarLander-v3") for _ in range(NUM_ENVS)])

state_size = env.observation_space.shape[0]
action_size = env.action_space.n


LEARNING_RATE = 0.001
DISCOUNT = 0.99
EPSILON = 1.0
EPSILON_DECAY = 0.995
EPSILON_MIN = 0.05
MEMORY_SIZE = 100000
BATCH_SIZE = 64
TARGET_UPDATE_FREQ = 1000
TOTAL_TIMESTEPS = 500000
TRAINING_START = 10000

# Experience Replay Memory
class ReplayBuffer:
    def __init__(self, size):
        self.buffer = deque(maxlen=size)

    def add(self, experience):
        self.buffer.append(experience)

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            torch.tensor(np.array(states), dtype=torch.float32, device=device),
            torch.tensor(actions, dtype=torch.int64, device=device),
            torch.tensor(rewards, dtype=torch.float32, device=device),
            torch.tensor(np.array(next_states), dtype=torch.float32, device=device),
            torch.tensor(dones, dtype=torch.float32, device=device),
        )

    def can_sample(self, batch_size):
        return len(self.buffer) >= batch_size

memory = ReplayBuffer(MEMORY_SIZE)

# Build Q-Network
class DQN(nn.Module):
    def __init__(self, state_size, action_size):
        super(DQN, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(state_size, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, action_size)
        )

    def forward(self, x):
        return self.network(x)

# Create two networks: online model and target model
model = DQN(state_size, action_size).to(device)
target_model = DQN(state_size, action_size).to(device)
target_model.load_state_dict(model.state_dict())  # Copy weights
target_model.eval()  # Target network is not trained

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
loss_fn = nn.MSELoss()

# Training function (Batch Training like SB3)
def train():
    if not memory.can_sample(BATCH_SIZE):
        return

    states, actions, rewards, next_states, dones = memory.sample(BATCH_SIZE)

    # Compute target Q-values
    with torch.no_grad():
        target_qs = target_model(next_states)
        max_future_q = torch.max(target_qs, dim=1)[0]
        new_qs = rewards + (1 - dones) * DISCOUNT * max_future_q

    # Get current Q-values and update only selected actions
    current_qs = model(states)
    current_qs = current_qs.gather(1, actions.unsqueeze(1)).squeeze(1)

    # Compute loss and update model
    loss = loss_fn(current_qs, new_qs)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item()  # Return loss value for tracking


# Training Loop (SB3-like strategy) with Metrics
state = env.reset()
state = torch.tensor(state, dtype=torch.float32, device=device)

step = 0
episode_rewards = []
episode_lengths = []
current_episode_reward = np.zeros(NUM_ENVS)
current_episode_length = np.zeros(NUM_ENVS)

while step < TOTAL_TIMESTEPS:
    if np.random.rand() < EPSILON:
        action = np.array([env.action_space.sample() for _ in range(NUM_ENVS)])  # Random actions
    else:
        with torch.no_grad():
            q_values = model(state)
            action = torch.argmax(q_values, dim=1).cpu().numpy()  # Select best actions

    # Take action and observe result
    next_state, reward, done, _ = env.step(action)
    next_state = torch.tensor(next_state, dtype=torch.float32, device=device)

    for i in range(NUM_ENVS):
        memory.add((state[i].cpu().numpy(), action[i], reward[i], next_state[i].cpu().numpy(), done[i]))

        current_episode_reward[i] += reward[i]
        current_episode_length[i] += 1

        if done[i]:  # End of episode
            episode_rewards.append(current_episode_reward[i])
            episode_lengths.append(current_episode_length[i])
            current_episode_reward[i] = 0
            current_episode_length[i] = 0

    state = next_state
    step += NUM_ENVS  # Increment step by number of envs

    # Train only after collecting enough data
    loss = None
    if step > TRAINING_START and step % 4 == 0:
        loss = train()

    # Update target network periodically
    if step % TARGET_UPDATE_FREQ == 0:
        target_model.load_state_dict(model.state_dict())

    # Decay exploration rate
    if EPSILON > EPSILON_MIN:
        EPSILON *= EPSILON_DECAY

    # Print progress every 10,000 steps
    if step % 10000 == 0:
        avg_reward = np.mean(episode_rewards[-100:]) if len(episode_rewards) > 0 else 0
        avg_length = np.mean(episode_lengths[-100:]) if len(episode_lengths) > 0 else 0
        last_loss = loss if loss is not None else 0

        print(f"Step: {step}, Epsilon: {EPSILON:.4f}, Avg Reward: {avg_reward:.2f}, Avg Length: {avg_length:.1f}, Loss: {last_loss:.4f}")

# Save trained model
# torch.save(model.state_dict(), "dqn_lunarlander_pytorch.pth")

print("-->>Training complete! Model saved as 'dqn_lunarlander_pytorch.pth'.")



Using device: cuda
Step: 10000, Epsilon: 0.0499, Avg Reward: -159.71, Avg Length: 77.7, Loss: 0.0000
Step: 20000, Epsilon: 0.0499, Avg Reward: -209.41, Avg Length: 91.2, Loss: 52.0185
Step: 30000, Epsilon: 0.0499, Avg Reward: -219.31, Avg Length: 132.1, Loss: 63.8663
Step: 40000, Epsilon: 0.0499, Avg Reward: -223.25, Avg Length: 215.0, Loss: 2.6963
Step: 50000, Epsilon: 0.0499, Avg Reward: -220.14, Avg Length: 295.8, Loss: 4.2607
Step: 60000, Epsilon: 0.0499, Avg Reward: -212.77, Avg Length: 388.4, Loss: 14.1633
Step: 70000, Epsilon: 0.0499, Avg Reward: -179.15, Avg Length: 489.4, Loss: 10.2360
Step: 80000, Epsilon: 0.0499, Avg Reward: -160.63, Avg Length: 560.9, Loss: 3.0446
Step: 90000, Epsilon: 0.0499, Avg Reward: -140.73, Avg Length: 674.0, Loss: 2.3601
Step: 100000, Epsilon: 0.0499, Avg Reward: -124.21, Avg Length: 745.1, Loss: 45.1614
Step: 110000, Epsilon: 0.0499, Avg Reward: -103.18, Avg Length: 814.9, Loss: 2.5404
Step: 120000, Epsilon: 0.0499, Avg Reward: -83.00, Avg Length: 

In [ ]:
import gymnasium as gym
import torch
import numpy as np
import imageio
from stable_baselines3.common.vec_env import DummyVecEnv

# Set up the environment (NO VECENV because we only need 1 environment for testing)
env = gym.make("LunarLander-v3", render_mode="rgb_array")  # Use "rgb_array" for video recording
state_size = env.observation_space.shape[0]
action_size = env.action_space.n

# Load the trained model
class DQN(torch.nn.Module):
    def __init__(self, state_size, action_size):
        super(DQN, self).__init__()
        self.network = torch.nn.Sequential(
            torch.nn.Linear(state_size, 128),
            torch.nn.ReLU(),
            torch.nn.Linear(128, 128),
            torch.nn.ReLU(),
            torch.nn.Linear(128, action_size)
        )

    def forward(self, x):
        return self.network(x)

# Load the model into GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DQN(state_size, action_size).to(device)
model.load_state_dict(torch.load("/content/dqn_lunarlander_pytorch.pth", map_location=device))
model.eval()  # Set model to evaluation mode

# Run the trained model & record gameplay
frames = []  # Store frames for the video
state = env.reset()[0]  # Get initial state
done = False
total_reward = 0

while not done:
    # Convert state to tensor
    state_tensor = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)

    # Select action from trained model
    with torch.no_grad():
        action = torch.argmax(model(state_tensor)).item()

    # Step environment
    next_state, reward, done, _, _ = env.step(action)
    total_reward += reward

    # Save frame for video
    frame = env.render()
    frames.append(frame)

    # Move to next state
    state = next_state

print(f"Test Episode Completed. Total Reward: {total_reward}")

# Save video using imageio
video_path = "lunarlander_test.mp4"
imageio.mimsave(video_path, frames, fps=30)
print(f"Video saved: {video_path}")